# Using `example_protobuf`

In this notebook, we show how to use the python package automatically created for the
protobuf message below:

In [1]:
from IPython.display import Markdown

with open("example_protobuf/protobuf/example_protobuf/person.proto") as f:
    text = f.read()

# Wrap in fenced code block for syntax-style formatting
Markdown(f"```proto\n{text}\n```")

```proto
syntax = "proto3";

package example_protobuf;

message Person {
  string name = 1;
  int64 age = 2;
  optional string email = 3;
  bool is_active = 4;

  message Address {
    string street = 1;
    string city = 2;
    int32 zip_code = 3;
  }

  enum Status {
    UNKNOWN = 0;
    ACTIVE = 1;
    INACTIVE = 2;
  }

  Address address = 5;
  repeated string tags = 6;
  Status status = 7;
  repeated Address previous_addresses = 8;
}

```

In [2]:
import polars as pl

# Load the Polars extension module
from example_protobuf.structpath import example_protobuf as mod_polars

# Load the Python bindings (created with `protoc`)
import example_protobuf.pybindings.person_pb2 as mod_python

## Create sample data

In [3]:
def create_samples(n: int) -> list[mod_python.Person]:
    """Create a list of Person messages for benchmarking"""

    persons = []

    for i in range(n):
        person = mod_python.Person()
        person.name = f"Person {i}"
        person.age = i % 80
        if i % 2 == 0:
            person.email = f"person{i}@example.com"
        person.is_active = i % 2 == 0

        person.address.street = f"Street {i}"
        person.address.city = f"City {i}"
        person.address.zip_code = i % 10000

        for j in range(i % 5):
            person.tags.append(f"tag{j}")

        person.status = mod_python.Person.Status.keys()[i % 3]

        for j in range(i % 3):
            person.previous_addresses.append(person.address)

        persons.append(person)

    return persons


samples = pl.DataFrame({"message": [p.SerializeToString() for p in create_samples(100000)]})
samples.head()

message
binary
"b""\x0a\x08Person\x200\x1a\x13person0@example.com\x20\x01*\x12\x0a\x08Street\x200\x12\x06City\x200"""
"b""\x0a\x08Person\x201\x10\x01*\x14\x0a\x08Street\x201\x12\x06City\x201\x18\x012\x04tag08\x01B\x14\x0a\x08Street\x201\x12\x06City""…"
"b""\x0a\x08Person\x202\x10\x02\x1a\x13person2@example.com\x20\x01*\x14\x0a\x08Street\x202\x12\x06City\x202\x18\x022\x04t""…"
"b""\x0a\x08Person\x203\x10\x03*\x14\x0a\x08Street\x203\x12\x06City\x203\x18\x032\x04tag02\x04tag12\x04tag2"""
"b""\x0a\x08Person\x204\x10\x04\x1a\x13person4@example.com\x20\x01*\x14\x0a\x08Street\x204\x12\x06City\x204\x18\x042\x04t""…"


## Benchmark python protobuf versus Polars package

In the sections below, we prove that we can replicate the values extracted using the python
protobuf implementation of the package above, using the polars plugin. We also compare
timings. Please do note that the Python protobuf implementation is extremely good performing,
so the time won with the plugin comes from the fact that multi-threading is used.

### Path `.name`

In [4]:
def f_python() -> pl.Series:
    return samples["message"].map_elements(lambda msg: mod_python.Person.FromString(msg).name).alias("name")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "name").alias("name"))["name"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

74.5 ms ± 7.3 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
17.6 ms ± 864 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Path `.age`

In [5]:
def f_python() -> pl.Series:
    return samples["message"].map_elements(lambda msg: mod_python.Person.FromString(msg).age).alias("age")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "age").alias("age"))["age"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

61.9 ms ± 1.17 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
15.6 ms ± 1.18 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Path `.email`

As the field is optional, the python implementation returns an empty string if missing.

In [6]:
def f_python() -> pl.Series:
    return samples["message"].map_elements(lambda msg: mod_python.Person.FromString(msg).email or None).alias("email")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "email").alias("email"))["email"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

65.2 ms ± 876 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)
17.9 ms ± 1.44 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Path `.is_active`

In [7]:
def f_python() -> pl.Series:
    return samples["message"].map_elements(lambda msg: mod_python.Person.FromString(msg).is_active).alias("is_active")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "is_active").alias("is_active"))["is_active"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

59.8 ms ± 898 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)
16.1 ms ± 651 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Path `.address`

In [8]:
def f_python() -> pl.Series:
    return_dtype = pl.Struct({"street": pl.String, "city": pl.String, "zip_code": pl.Int32})
    return samples["message"].map_elements(
        lambda msg: {
            "street": (address := mod_python.Person.FromString(msg).address).street,
            "city": address.city,
            "zip_code": address.zip_code
        },
        return_dtype=return_dtype
    ).alias("address")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "address").alias("address"))["address"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

153 ms ± 10.3 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
55.8 ms ± 3.98 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Path `.address.street`

In [9]:
def f_python() -> pl.Series:
    return samples["message"].map_elements(lambda msg: mod_python.Person.FromString(msg).address.street).alias("address_street")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "address.street").alias("address_street"))["address_street"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

80.3 ms ± 948 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)
24 ms ± 2.69 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Path `.tags`

Note that the repeated `tags` field has a special protobuf type (`RepeatedScalarContainer`).
The conversion to a python iterable is needed, and takes a very significant amount of time.

In [10]:
def f_python() -> pl.Series:
    return_dtype = pl.List(pl.String)
    return samples["message"].map_elements(
        lambda msg: list(mod_python.Person.FromString(msg).tags),
        return_dtype=return_dtype
    ).alias("tags")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "tags").alias("tags"))["tags"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

375 ms ± 8.5 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
60.6 ms ± 7.49 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Path `.tags[0]`

In [11]:
def f_python() -> pl.Series:
    return samples["message"].map_elements(lambda msg: x[0] if len(x := mod_python.Person.FromString(msg).tags) else None).alias("tags_0")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "tags[0]").alias("tags_0"))["tags_0"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

84.3 ms ± 1.42 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
17.4 ms ± 877 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Path `.status`

The Python protobuf implementation returns integers for enums, while the Polars plugin
returns an Enum type. 

In [12]:
def f_python() -> pl.Series:
    return samples["message"].map_elements(
        lambda msg: mod_python.Person.Status.Name(mod_python.Person.FromString(msg).status),
        return_dtype=pl.Enum(list(mod_python.Person.Status.keys()))
    ).alias("status")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "status").alias("status"))["status"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

87.7 ms ± 826 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)
21.7 ms ± 607 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Path `.previous_addresses`

In [13]:
def f_python() -> pl.Series:
    return_dtype = pl.List(pl.Struct({"street": pl.String, "city": pl.String, "zip_code": pl.Int32}))
    return samples["message"].map_elements(
        lambda msg: [
            {"street": address.street, "city": address.city, "zip_code": address.zip_code}
            for address in mod_python.Person.FromString(msg).previous_addresses
        ],
        return_dtype=return_dtype
    ).alias("address")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "previous_addresses").alias("previous_addresses"))["previous_addresses"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

441 ms ± 17.3 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
437 ms ± 18.9 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Path `.previous_addresses[0].street`

In [14]:
def f_python() -> pl.Series:
    return samples["message"].map_elements(
        lambda msg: addresses[0].street if len(addresses := mod_python.Person.FromString(msg).previous_addresses) else None,
    ).alias("previous_addresses_0_street")

def f_polars() -> pl.Series:
    return samples.select(mod_polars.Person.get_value(pl.col("message"), "previous_addresses[0].street").alias("previous_addresses_0_street"))["previous_addresses_0_street"]

assert f_python().equals(f_polars())
%timeit -n 1 -r 10 f_python()
%timeit -n 1 -r 10 f_polars()

91.5 ms ± 2.65 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
21.6 ms ± 1.23 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
